# Exporting a PyTorch model for use with simple_triton

In this notebook, we will export a PyTorch model for use with simple_triton/tritonserver running with a PyTorch backend. This is useful if you find or train your own model in PyTorch format that you want to use with tritonserver.
Primarily these steps is about giving information about our model's input and output shapes in a format that Tritonserver can understand.

We will:
1. Create a dummy PyTorch model (or load your own model from a checkpoint)
2. Save the model into the tritonserver models directory
3. Set model options
4. Launch tritonserver with our model
5. Load the model with the client

We also include a cell to do inference with the model, but this is optional.


## Import needed libraries
You might have to install torch:
`pip3 install --no-cache-dir torch torchvision --index-url https://download.pytorch.org/whl/cpu`

In [53]:
import glob
import os
from pprint import pprint

# hide the GPU from TensorFlow - tritonserver will use it, not us
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import torch
from jupyter_core.utils import ensure_dir_exists

from simple_triton.feature_extraction import inference
from simple_triton.utils import analyze
from simple_triton.model import TritonModel


# Step 1: Create a dummy model

If you want to use your own model instead, load it here. The model should be in eval mode, and the input and output shapes should be known.
Otherwise, we will create a simple linear model with 3 input features and 1 output feature.

In [54]:
model_name = "my_model"
model = torch.nn.Sequential(
    torch.nn.Linear(3, 1),
)
model.eval()
# simple test of the model
test_data = torch.rand(3, 3)
model(test_data).detach().numpy()

array([[0.20215343],
       [0.20523736],
       [0.28746206]], dtype=float32)

# Step 2: Save model into tritonserver models directory
The minimal directory structure for a model is:
```
models/
`-- model_name
    |-- 1
    |   `-- [model.pt]
```
For more information, see: https://github.com/triton-inference-server/pytorch_backend

The below code will create the model in the correct directory structure, saved in `/tmp`:

In [55]:
username = os.environ.get("USER", "user")
if not username: username = "defaultuser"
model_dir = f"/tmp/models_{username}"
dst_dir = f"{model_dir}/{model_name}/1/model.pt"
ensure_dir_exists(os.path.dirname(dst_dir))
script_model = torch.jit.script(model)
torch.jit.save(script_model, dst_dir)


# Step 3: Setting model options
For PyTorch models, tritonserver needs a little help to understand the input dimensions of the model (how many features are in each sample) and output dimensions. We have two options:
1. Generate a "config.pbtxt", which will be placed in the "model_name" directory
2. Export the model as ONNX, which lets tritonserver automatically configure the model itself

This notebook will do both. If you only want to use one, just comment out the other.

### Option 1: Obtaining the config.pbtxt file
The easiest method is to use chatGPT to generate the config.pbtxt file. Just send it your model code and ask for a config.pbtxt for tritonserver.

Otherwise, below is an example of a config.pbtxt file for the model we just created, which will also be automatically put in the correct directory.


In [56]:
cfg='''
name: "your_model_name"
platform: "pytorch_libtorch"
max_batch_size: 1000000
input [
  {
    name: "input_1"
    data_type: TYPE_FP32
    dims: [3]
  }
]
output [
  {
    name: "output_1"
    data_type: TYPE_FP32
    dims: [1]
  }
]'''.replace("your_model_name", model_name)
config_dst = os.path.join(model_dir, model_name, "config.pbtxt")
with open(config_dst, "w") as f:
    f.write(cfg)

### Option 2: exporting the model as ONNX
ONNX is a standard format for neural networks that is supported by tritonserver (and many other frameworks). It is a good choice if you want to avoid the hassle of creating a config.pbtxt file, and it ensures that your model is easy to use by other developers.

In [72]:
# 3 = number of features from the first linear layer that we defined earlier in this notebook
model_inputs = torch.rand(1, 3)
# enable dynamic batch sizes
dynamic_axes = {'input': {0: 'batch'}, 'output': {0: 'batch'}}

dst_onnx = os.path.join(model_dir, f"{model_name}.onnx", "1", "model.onnx")
ensure_dir_exists(os.path.dirname(dst_onnx))

torch.onnx.export(model,
                  (model_inputs,),
                  dst_onnx,
                  dynamic_axes=dynamic_axes,
                  input_names=["input"],
                  output_names=["output"],
                  )

In [58]:
# print directory structure of the model dir
glob.glob(os.path.join(model_dir, "**"), recursive=True)

['/tmp/models_aza4423/',
 '/tmp/models_aza4423/my_model',
 '/tmp/models_aza4423/my_model/1',
 '/tmp/models_aza4423/my_model/1/model.pt',
 '/tmp/models_aza4423/my_model/config.pbtxt',
 '/tmp/models_aza4423/my_model.onnx',
 '/tmp/models_aza4423/my_model.onnx/1',
 '/tmp/models_aza4423/my_model.onnx/1/model.onnx']

If you want to, you can now copy the model to the tritonserver models directory, with a command like: `cp -r /tmp/models_${USER:-defaultuser} /path/to/tritonserver/models`. Otherwise, the below commands will work just fine.

# Step 4: Launching Tritonserver
```bash
docker run --name tritonserver_$USER --gpus all --rm -p 8001:8001 --shm-size=1g --ulimit memlock=-1 --ipc=host -v /tmp/models_${USER:-defaultuser}:/models nvcr.io/nvidia/tritonserver:24.07-py3  tritonserver --model-repository=/models --model-control-mode=explicit --exit-on-error=false --strict-model-config=false --backend-config=default-max-batch-size=32 --metrics-config summary_latencies=true
```

## Optional: load the model without the client
if you want to, you can also make Tritonserver load the model right away on boot by adding `--load-model <model_name>` to the command above. If using ONNX, add ".onnx" to the model name.
```bash
docker run --name tritonserver_$USER --gpus all --rm -p 8001:8001 --shm-size=1g --ulimit memlock=-1 --ipc=host -v /tmp/models_${USER:-defaultuser}:/models nvcr.io/nvidia/tritonserver:24.07-py3  tritonserver --model-repository=/models --model-control-mode=explicit --exit-on-error=false --strict-model-config=false --backend-config=default-max-batch-size=32 --metrics-config summary_latencies=true --load-model my_model --load-model my_model.onnx
```
If this works, you should see something like this in the console:
```bash
+---------------+---------+--------+
| Model         | Version | Status |
+---------------+---------+--------+
| my_model      | 1       | READY  |
| my_model.onnx | 1       | READY  |
+---------------+---------+--------+
```

# Step 5: Loading the model with the client
The below code can be used to load the model, regardless of whether it has been loaded already or not:

In [79]:
url = "localhost:8001/"

# if using ONNX, add ".onnx" to the model name
srv_model = TritonModel(model_name, url)
srv_model.load()
assert srv_model.is_loaded()
print("###### Pytorch model loaded ######")
pprint(srv_model.get_config())

print()
print()
print("###### Pytorch ONNX model loaded ######")
srv_model_onnx = TritonModel(model_name + ".onnx", url)
srv_model_onnx.load()
assert srv_model_onnx.is_loaded()
pprint(srv_model_onnx.get_config())


###### Pytorch model loaded ######
{'backend': 'pytorch',
 'defaultModelFilename': 'model.pt',
 'input': [{'dataType': 'TYPE_FP32', 'dims': ['3'], 'name': 'input_1'}],
 'instanceGroup': [{'count': 1,
                    'gpus': [0, 1, 2, 3, 4, 5, 6],
                    'kind': 'KIND_GPU',
                    'name': 'my_model'}],
 'maxBatchSize': 1000000,
 'name': 'my_model',
 'optimization': {'inputPinnedMemory': {'enable': True},
                  'outputPinnedMemory': {'enable': True}},
 'output': [{'dataType': 'TYPE_FP32', 'dims': ['1'], 'name': 'output_1'}],
 'platform': 'pytorch_libtorch',
 'versionPolicy': {'latest': {'numVersions': 1}}}


###### Pytorch ONNX model loaded ######
{'backend': 'onnxruntime',
 'defaultModelFilename': 'model.onnx',
 'dynamicBatching': {'preferredBatchSize': [32]},
 'input': [{'dataType': 'TYPE_FP32', 'dims': ['3'], 'name': 'input'}],
 'instanceGroup': [{'count': 1,
                    'gpus': [0, 1, 2, 3, 4, 5, 6],
                    'kind': 'KIND_

### For more on inference...
See the other notebooks in `examples` directory.

Anyhow, below is a simple demonstration where we send a single batch of three "images" with metadata:

In [82]:
data = [(np.array(
    [ # first "image" batch
        [1,2,3], # first "image"
        [4,5,6], # second "image"
        [7,8,9], # third "image"
    ], dtype=np.float32),
    [ # first "metadata" batch
     {"meta_key": "meta_value"}, # first "image" metadata
     {"meta_key": "meta_value"}, # second "image" metadata
     {"meta_key": "meta_value"}, # third "image" metadata
 ])]
print(data)

features, tile_info, times, failures = inference(
    iter(data),
    model_name,
    source=None,
    target=None,
    url=url,
    limit=1,
)
assert not failures
analyze(times)

print(f"Inference done. Output has shape: {features.shape}")

features_onnx, tile_info, times, failures = inference(
    iter(data),
    model_name + ".onnx",
    source=None,
    target=None,
    url=url,
    limit=1,
)
assert not failures
analyze(times)

# output is per batch, flatten the array
features_onnx = np.concatenate(features_onnx[0], axis=0)
print(f"Inference done for ONNX. Output has shape: {features_onnx.shape}")

assert np.array_equal(features, features_onnx)

[(array([[1., 2., 3.],
       [4., 5., 6.],
       [7., 8., 9.]], dtype=float32), [{'meta_key': 'meta_value'}, {'meta_key': 'meta_value'}, {'meta_key': 'meta_value'}])]


Batches: 1it [00:00, 51.09it/s]


Inference done. Output has shape: (3, 1)


Batches: 1it [00:00, 152.69it/s]

Inference done for ONNX. Output has shape: (3, 1)
